In [1]:
!hostname

gpua087.delta.ncsa.illinois.edu


In [2]:
import scanpy as sc
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
ad.settings.allow_write_nullable_strings = True
import matplotlib.pyplot as plt
import seaborn as sns
import gseapy as gp
import decoupler

/projects/bhdw/asachan/.conda/envs/decoupler_psc/lib/python3.12/site-packages/scanpy/_utils/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/projects/bhdw/asachan/.conda/envs/decoupler_psc/lib/python3.12/site-packages/scanpy/__init__.py:36: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/projects/bhdw/asachan/.conda/envs/decoupler_psc/lib/python3.12/site-packages/scanpy/readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [4]:
import os
os.chdir('/projects/bhdw/asachan/methods/maxtoki-perturb/scripts/torch_pipeline/data_filter')  # directory containing utils.py
import sys
import logging
import warnings

export_dir = "/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human"
raw_dir = "/work/hdd/bgdb/asachan/datasets_hdd/SKM_ageing_human/raw_files"

In [5]:
from utils import *

In [6]:
adata_rna = '/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/rna_objects/rna_female_type2_ds_wrt_HALLMARK_DNA_REPAIR.h5ad'
out_tmp = '/projects/bhdw/asachan/tmp'

In [7]:
rna_adata = sc.read_h5ad(adata_rna)

In [8]:
rna_adata

AnnData object with n_obs × n_vars = 4111 × 48355
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII', 'bead_count', 'age_categorical', 'GOBP_DNA_DAMAGE_RESPONSE', 'GOBP_DNA_REPAIR', 'HALLMARK_DNA_REPAIR', 'REACTOME_DNA_REPAIR'
    var: 'features', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'neighbors', 'pca', 'rank_genes_groups', 'sample_colors', 'umap'
    obsm: 'X_pca', 'X_umap', 'score_aucell'
    varm: 'PCs'
    layers: 'counts', 'counts_float', 'lognorm'
    obsp: 'connectivities', 'distances'

In [9]:
rna_adata.var

,features,highly_variable,means,dispersions,dispersions_norm
WASH7P,WASH7P,False,3.645889e-02,1.298698,-0.149975
CICP27,CICP27,False,4.044329e-03,1.640501,0.720140
AL732372.2,AL732372.2,False,4.103191e-02,1.327958,-0.075487
AL669831.3,AL669831.3,True,2.932112e-01,1.841465,2.051180
MTND2P28,MTND2P28,False,4.439346e-02,1.263465,-0.239666
...,...,...,...,...,...
AC063955.1,AC063955.1,False,1.000000e-12,NaN,NaN
MKNK2P1,MKNK2P1,False,1.000000e-12,NaN,NaN
AC087072.1,AC087072.1,False,1.000000e-12,NaN,NaN
CDC42P2,CDC42P2,False,1.000000e-12,NaN,NaN


In [7]:
rna_adata.obs['sample'].value_counts()

sample
YM2    1989
OM9    1278
OM6     722
P26     122
Name: count, dtype: int64

In [8]:
rna_adata.obs['age'].value_counts()

age
80    2000
34    1989
17     122
Name: count, dtype: int64

In [10]:
rna_adata.obs['nCount_RNA']

CELL303_N2_1_1_6_1     3136.2871
CELL2900_N1_1_1_6_1    2202.0882
CELL2202_N1_1_1_6_1    2602.6083
CELL3194_N1_1_1_6_1    2387.1073
CELL523_N2_1_1_6_1     2973.4530
                         ...    
CELL1197_N1_1_11_1     2199.7543
CELL1397_N1_1_11_1     2185.8620
CELL350_N1_1_11_1      2451.8629
CELL1777_N1_1_11_1     2096.1938
CELL1865_N1_1_11_1     2289.9488
Name: nCount_RNA, Length: 4111, dtype: float64

### add emsembl_ids to gene name for tokenization in maxtoki